# ⭐ Day 38: Feature Distribution & Skewness Analysis
## Understanding Data Shape for Better AI & ML Models

**Day 38 of 369-day Python & AI Learning Path** 🚀


## Welcome to Day 38!

Welcome back to our Python & AI Learning Path! Today we are diving deep into a foundational aspect of data preprocessing that often makes the difference between mediocre and excellent models: **feature distribution and skewness analysis**. While it is tempting to feed raw data directly into algorithms, understanding the shape of your distributions is a hallmark of experienced data scientists.

Feature distributions tell the story of your data. They reveal whether variables follow normal patterns, exhibit extreme tails, or cluster around specific values. Skewed distributions—where data piles up on one side with a long tail stretching to the other—are particularly problematic for many machine learning algorithms. Linear models, neural networks, and distance-based methods like KNN and SVM all assume or perform better with normally distributed features. When this assumption is violated, you risk biased parameter estimates, slower convergence, and suboptimal performance.

Beyond model performance, distribution analysis guides critical preprocessing decisions. Should you log-transform that income variable? Is a square root transformation sufficient for the area feature? Or do you need the power of Box-Cox to find the optimal transformation? These choices directly impact how your model learns from the data. Additionally, understanding your target variable distribution is crucial for regression tasks—predicting skewed targets without transformation can lead to models that systematically underpredict high values.

In this notebook, we will learn to visualize distributions, quantify skewness and kurtosis, and apply various transformation techniques to normalize skewed features. By the end, you will have a systematic approach to distribution analysis that improves model performance across a wide range of algorithms. Let us shape up your data! 📈


## 📋 Table of Contents

1. [Why Feature Distributions Matter in Machine Learning](#1-why-feature-distributions-matter-in-machine-learning)
2. [Understanding Skewness](#2-understanding-skewness)
3. [Loading Dataset and Selecting Numerical Features](#3-loading-dataset-and-selecting-numerical-features)
4. [Visualizing Distributions](#4-visualizing-distributions)
5. [Calculating Skewness and Kurtosis](#5-calculating-skewness-and-kurtosis)
6. [Interpreting Skewness Values](#6-interpreting-skewness-values)
7. [Transformation Techniques to Reduce Skewness](#7-transformation-techniques-to-reduce-skewness)
8. [Analyzing Distribution of Target Variable](#8-analyzing-distribution-of-target-variable)
9. [Key Insights and Preprocessing Recommendations](#9-key-insights-and-preprocessing-recommendations)
10. [🛠️ Hands-On Exercises](#-hands-on-exercises)
11. [Solutions & Key Insights](#-solutions--key-insights)


## 1. Why Feature Distributions Matter in Machine Learning 📈

### The Distribution-Performance Connection

Many ML algorithms make implicit or explicit assumptions about data distributions:

- **📊 Linear Models** (Linear Regression, Logistic Regression): Assume normally distributed errors; features with extreme skew can bias coefficient estimates
- **🤖 Neural Networks**: Gradient descent converges faster with normalized, symmetric distributions
- **📏 Distance-Based Methods** (KNN, SVM, K-Means): Skewed features dominate distance calculations
- **🌳 Tree-Based Models**: Less sensitive to skewness but can still benefit from transformations

### Consequences of Ignoring Skewness:

| Problem | Cause | Solution |
|---------|-------|----------|
| Biased predictions | Algorithm learns from tail outliers | Transform to reduce tail influence |
| Poor convergence | Gradient updates dominated by extreme values | Normalize distributions |
| Feature dominance | One skewed feature overwhelms others | Apply scaling + transformation |
| Invalid statistics | Mean/median no longer representative | Use robust statistics or transform |

### Key Principle
**Distribution analysis should be the first step after data cleaning.** Understanding shape guides all subsequent preprocessing decisions and helps set realistic model performance expectations.


## 2. Understanding Skewness 📉

### What is Skewness?
Skewness measures the asymmetry of the probability distribution of a real-valued random variable about its mean.

### Types of Skewness:

#### 🔹 Positive Skew (Right-Skewed)
- Tail extends to the **right** (higher values)
- Mean > Median > Mode
- Common in: Income, house prices, population counts, transaction amounts
- **Example**: Most people earn modest incomes, few earn millions

#### 🔹 Negative Skew (Left-Skewed)
- Tail extends to the **left** (lower values)
- Mean < Median < Mode
- Common in: Age at retirement, test scores (easy exams), golf scores
- **Example**: Most students score high on easy tests, few score very low

#### 🔹 Zero Skew (Symmetric)
- Normal distribution
- Mean = Median = Mode
- Ideal for most ML algorithms

### Visual Guide:
```
Negative Skew          Symmetric           Positive Skew
     ←╲                ╱╲                    ╱╲
       ╲              ╱  ╲                  ╱  ╲→
        ╲            ╱    ╲                ╱    ╲
         ╲__________╱      ╲______________╱      ╲__________
     Low scores    High   Balanced       Low    High    Very High
```

**💡 Remember**: Positive skew means positive tail (right side), negative skew means negative tail (left side)!


In [ ]:
# Import essential libraries for distribution analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import skew, kurtosis, boxcox, yeojohnson
import warnings
warnings.filterwarnings('ignore')

# Set visualization parameters
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)
np.random.seed(42)

print('✅ Libraries loaded successfully!')
print(f'NumPy version: {np.__version__}')
print(f'Pandas version: {pd.__version__}')
print(f'Scipy stats available for transformations')


In [ ]:
# Create a synthetic dataset with varied skewness patterns
# Simulating a real estate / customer transaction dataset

n_samples = 1000

# Feature 1: House prices (highly right-skewed, log-normal)
house_price = np.random.lognormal(12, 0.5, n_samples)

# Feature 2: Income (right-skewed, Pareto-like)
income = np.random.pareto(2, n_samples) * 50000 + 20000

# Feature 3: Age (approximately normal, slightly left-skewed)
age = np.random.normal(40, 12, n_samples).clip(18, 80)

# Feature 4: Transaction amount (highly skewed, few big spenders)
transaction_amount = np.random.exponential(100, n_samples) ** 1.5

# Feature 5: Distance to city center (right-skewed, most close, few far)
distance_to_city = np.random.exponential(10, n_samples).clip(0.5, 100)

# Feature 6: Credit score (left-skewed, most good scores)
credit_score = 850 - np.random.exponential(100, n_samples).clip(0, 550)

# Feature 7: Website visits (count data, right-skewed)
website_visits = np.random.poisson(3, n_samples) ** 2

# Feature 8: Satisfaction score (negative skew, most satisfied)
satisfaction = 10 - np.random.beta(2, 5, n_samples) * 10

# Feature 9: Square footage (moderately right-skewed)
square_feet = np.random.lognormal(7.5, 0.4, n_samples)

# Feature 10: Years at current job (right-skewed, few long-tenured)
job_tenure = np.random.exponential(3, n_samples).clip(0, 40)

# Assemble dataframe
df = pd.DataFrame({
    'house_price': house_price,
    'income': income,
    'age': age,
    'transaction_amount': transaction_amount,
    'distance_to_city': distance_to_city,
    'credit_score': credit_score,
    'website_visits': website_visits,
    'satisfaction': satisfaction,
    'square_feet': square_feet,
    'job_tenure': job_tenure
})

# Create target variable with some relationships
target = (
    np.log(house_price) * 5000 +
    np.log(income + 1) * 2000 -
    distance_to_city * 100 +
    credit_score * 50 +
    np.random.normal(0, 50000, n_samples)
)
df['target'] = target

print('✅ Synthetic dataset created with varied skewness patterns!')
print(f'Dataset shape: {df.shape}')
print(f'Features: {len(df.columns) - 1} (excluding target)')
print(f'\nFirst 5 rows:')
print(df.head())


## 3. Loading Dataset and Selecting Numerical Features 📋

Let us examine our dataset structure and identify numerical features for distribution analysis.


In [ ]:
# Dataset overview and feature selection
print('=' * 70)
print('📊 DATASET OVERVIEW')
print('=' * 70)

print(f'Total samples: {len(df):,}')
print(f'Total features: {len(df.columns) - 1} (excluding target)')
print(f'Target variable: target')

# Identify numerical features
numerical_features = df.select_dtypes(include=[np.number]).columns.tolist()
numerical_features.remove('target')  # Exclude target for now

print(f'\nNumerical features for analysis: {len(numerical_features)}')
for i, feat in enumerate(numerical_features, 1):
    print(f'  {i:2d}. {feat}')

# Basic statistics
print(f'\nBasic Statistics:')
print(df[numerical_features].describe().round(2))

# Check for missing values
missing = df[numerical_features].isnull().sum()
if missing.sum() == 0:
    print(f'\n✅ No missing values in numerical features')
else:
    print(f'\n⚠️ Missing values detected:')
    print(missing[missing > 0])


## 4. Visualizing Distributions 📊

Visual inspection is the first and most important step in distribution analysis.


In [ ]:
# Create comprehensive distribution visualizations
fig, axes = plt.subplots(5, 2, figsize=(14, 20))
axes = axes.flatten()

for idx, feature in enumerate(numerical_features):
    ax = axes[idx]
    
    # Histogram with KDE
    sns.histplot(df[feature], kde=True, ax=ax, color='steelblue', alpha=0.7, edgecolor='black')
    
    # Add mean and median lines
    mean_val = df[feature].mean()
    median_val = df[feature].median()
    ax.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.1f}')
    ax.axvline(median_val, color='green', linestyle='--', linewidth=2, label=f'Median: {median_val:.1f}')
    
    # Calculate skewness
    skew_val = skew(df[feature])
    
    ax.set_title(f'{feature.replace("_", " ").title()}\nSkewness: {skew_val:.2f}', 
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('📊 Feature Distributions with KDE Curves\n(Red=Mean, Green=Median)', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('💡 Insight: Compare mean vs median positions—larger gaps indicate higher skewness!')


In [ ]:
# Boxplots and violin plots for outlier and distribution shape analysis
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Boxplot
ax1 = axes[0]
df[numerical_features].boxplot(ax=ax1, rot=45)
ax1.set_title('📦 Boxplots: Outlier Detection and Distribution Spread', 
              fontsize=12, fontweight='bold')
ax1.set_ylabel('Value')
ax1.grid(axis='y', alpha=0.3)

# Violin plot
ax2 = axes[1]
df_melted = df[numerical_features].melt(var_name='Feature', value_name='Value')
sns.violinplot(data=df_melted, x='Feature', y='Value', ax=ax2, palette='Set2')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha='right')
ax2.set_title('🎻 Violin Plots: Distribution Shape and Density', 
              fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print('💡 Insight: Boxplots show outliers and IQR; violin plots reveal multimodal distributions!')


In [ ]:
# Q-Q plots for normality assessment
fig, axes = plt.subplots(5, 2, figsize=(14, 20))
axes = axes.flatten()

for idx, feature in enumerate(numerical_features):
    ax = axes[idx]
    
    # Q-Q plot against normal distribution
    stats.probplot(df[feature], dist='norm', plot=ax)
    
    ax.set_title(f'Q-Q Plot: {feature.replace("_", " ").title()}\n(Deviations from red line = non-normality)', 
                 fontsize=10, fontweight='bold')
    ax.grid(alpha=0.3)

plt.suptitle('📈 Q-Q Plots: Normality Assessment\n(Straight line = normal distribution)', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('💡 Insight: Curved Q-Q plots indicate skewness; S-shapes indicate heavy/light tails!')


## 5. Calculating Skewness and Kurtosis 🧮

Quantify distribution shape with statistical measures.


In [ ]:
# Comprehensive skewness and kurtosis analysis
print('=' * 70)
print('📊 SKEWNESS AND KURTOSIS ANALYSIS')
print('=' * 70)

def calculate_moments(data):
    """Calculate skewness and kurtosis with interpretation"""
    skew_val = skew(data)
    kurt_val = kurtosis(data, fisher=True)  # Excess kurtosis (normal = 0)
    
    # Interpret skewness
    if abs(skew_val) < 0.5:
        skew_interpret = 'Approximately symmetric'
    elif abs(skew_val) < 1:
        skew_interpret = 'Moderately skewed'
    else:
        skew_interpret = 'Highly skewed'
    
    # Interpret kurtosis
    if kurt_val > 1:
        kurt_interpret = 'Leptokurtic (heavy tails)'
    elif kurt_val < -1:
        kurt_interpret = 'Platykurtic (light tails)'
    else:
        kurt_interpret = 'Mesokurtic (normal-like tails)'
    
    return skew_val, kurt_val, skew_interpret, kurt_interpret

# Calculate for all features
moments_data = []
for feature in numerical_features:
    skew_val, kurt_val, skew_int, kurt_int = calculate_moments(df[feature])
    moments_data.append({
        'Feature': feature,
        'Skewness': round(skew_val, 3),
        'Kurtosis': round(kurt_val, 3),
        'Skew_Interpretation': skew_int,
        'Kurt_Interpretation': kurt_int
    })

moments_df = pd.DataFrame(moments_data)
moments_df = moments_df.sort_values('Skewness', key=abs, ascending=False)

print('\nSkewness and Kurtosis Summary (sorted by absolute skewness):')
print(moments_df.to_string(index=False))

# Summary statistics
high_skew = moments_df[abs(moments_df['Skewness']) > 1]
moderate_skew = moments_df[(abs(moments_df['Skewness']) >= 0.5) & (abs(moments_df['Skewness']) <= 1)]
low_skew = moments_df[abs(moments_df['Skewness']) < 0.5]

print(f'\n📈 Distribution Summary:')
print(f'   Highly skewed features (|skew| > 1): {len(high_skew)}')
print(f'   Moderately skewed features (0.5 <= |skew| <= 1): {len(moderate_skew)}')
print(f'   Approximately symmetric features (|skew| < 0.5): {len(low_skew)}')


## 6. Interpreting Skewness Values 💡

Understanding what skewness values mean for your ML pipeline.

### 📏 Skewness Interpretation Rules

| Skewness Value | Interpretation | ML Impact | Action Needed |
|----------------|----------------|-----------|---------------|
| -0.5 to +0.5 | Fairly symmetric | ✅ Minimal | None |
| -1.0 to -0.5 or +0.5 to +1.0 | Moderate skew | ⚠️ Moderate | Consider transformation |
| < -1.0 or > +1.0 | High skew | ❌ Significant | Strongly recommend transformation |

### 🎯 Algorithm-Specific Sensitivity

| Algorithm | Skewness Sensitivity | Reason |
|-----------|---------------------|--------|
| Linear Regression | High | Assumes normal errors |
| Logistic Regression | High | Assumes linear log-odds |
| Neural Networks | Moderate | Gradient flow affected |
| KNN, SVM | High | Distance metrics distorted |
| Decision Trees | Low | Split-based, distribution agnostic |
| Random Forest, XGBoost | Very Low | Ensemble of trees |
| Naive Bayes | High | Assumes feature independence with specific distributions |

### 📊 Kurtosis Interpretation
| Kurtosis | Interpretation | Impact |
|----------|----------------|--------|
| > 3 (or > 0 excess) | Leptokurtic | Heavy tails, more outliers |
| ≈ 3 (or ≈ 0 excess) | Mesokurtic | Normal-like tails |
| < 3 (or < 0 excess) | Platykurtic | Light tails, fewer outliers |

**💡 Key Takeaway**: Focus on skewness for transformation decisions; kurtosis guides outlier handling strategies.


In [ ]:
# Visualize skewness comparison across features
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Skewness bar chart
ax1 = axes[0]
skew_values = moments_df.sort_values('Skewness', ascending=True)
colors = ['red' if abs(x) > 1 else 'orange' if abs(x) > 0.5 else 'green' for x in skew_values['Skewness']]
bars = ax1.barh(skew_values['Feature'], skew_values['Skewness'], color=colors, alpha=0.7, edgecolor='black')
ax1.axvline(x=0, color='black', linewidth=1)
ax1.axvline(x=0.5, color='orange', linestyle='--', linewidth=1.5, label='Moderate threshold')
ax1.axvline(x=-0.5, color='orange', linestyle='--', linewidth=1.5)
ax1.axvline(x=1, color='red', linestyle='--', linewidth=1.5, label='High threshold')
ax1.axvline(x=-1, color='red', linestyle='--', linewidth=1.5)
ax1.set_xlabel('Skewness', fontsize=11)
ax1.set_title('📊 Skewness by Feature\n(Green=OK, Orange=Moderate, Red=High)', 
              fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(axis='x', alpha=0.3)

# Add value labels
for i, (idx, row) in enumerate(skew_values.iterrows()):
    ax1.text(row['Skewness'] + 0.05 if row['Skewness'] > 0 else row['Skewness'] - 0.05, 
             i, f'{row["Skewness"]:.2f}', 
             va='center', ha='left' if row['Skewness'] > 0 else 'right', fontsize=9)

# Kurtosis bar chart
ax2 = axes[1]
kurt_values = moments_df.sort_values('Kurtosis', ascending=True)
colors_kurt = ['red' if x > 3 else 'orange' if x > 1 else 'green' if x > -1 else 'blue' for x in kurt_values['Kurtosis']]
bars2 = ax2.barh(kurt_values['Feature'], kurt_values['Kurtosis'], color=colors_kurt, alpha=0.7, edgecolor='black')
ax2.axvline(x=0, color='black', linewidth=1, label='Normal (excess=0)')
ax2.set_xlabel('Excess Kurtosis', fontsize=11)
ax2.set_title('📈 Kurtosis by Feature\n(Green=Normal-like, Red=Heavy tails)', 
              fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print('✅ Skewness and kurtosis visualization complete!')


## 7. Transformation Techniques to Reduce Skewness 🔄

Apply mathematical transformations to normalize skewed distributions.


In [ ]:
# Transformation functions
def apply_transformations(data):
    """Apply various transformations and return results"""
    results = {}
    
    # Original
    results['original'] = data
    
    # Log transformation (handles positive skew)
    # Add small constant to handle zeros
    min_val = data.min()
    offset = abs(min_val) + 1 if min_val <= 0 else 0
    results['log'] = np.log(data + offset)
    
    # Square root transformation (milder than log)
    results['sqrt'] = np.sqrt(data + offset)
    
    # Box-Cox transformation (finds optimal lambda)
    # Only works for positive data
    if min_val > 0:
        results['boxcox'], fitted_lambda = boxcox(data)
        results['boxcox_lambda'] = fitted_lambda
    else:
        # Shift to positive for Box-Cox
        shifted_data = data + offset
        results['boxcox'], fitted_lambda = boxcox(shifted_data)
        results['boxcox_lambda'] = fitted_lambda
    
    # Yeo-Johnson (handles negative values)
    results['yeojohnson'], fitted_lambda_yj = yeojohnson(data)
    results['yeojohnson_lambda'] = fitted_lambda_yj
    
    return results

# Select most skewed feature for demonstration
most_skewed_feature = moments_df.iloc[0]['Feature']
print(f'🔄 Demonstrating transformations on most skewed feature: {most_skewed_feature}')
print(f'   Original skewness: {moments_df.iloc[0]["Skewness"]:.3f}')

# Apply transformations
transformed_data = apply_transformations(df[most_skewed_feature])

# Calculate skewness for each transformation
print(f'\n📊 Skewness after transformations:')
for transform_name in ['original', 'log', 'sqrt', 'boxcox', 'yeojohnson']:
    if transform_name in transformed_data:
        skew_val = skew(transformed_data[transform_name])
        print(f'   {transform_name:12s}: {skew_val:>7.3f}')


In [ ]:
# Visualize before/after transformations
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

transformations = ['original', 'log', 'sqrt', 'boxcox', 'yeojohnson']
titles = ['Original', 'Log Transform', 'Square Root', 'Box-Cox', 'Yeo-Johnson']

for idx, (transform, title) in enumerate(zip(transformations, titles)):
    if idx >= len(axes):
        break
        
    ax = axes[idx]
    data = transformed_data[transform]
    
    # Histogram with KDE
    sns.histplot(data, kde=True, ax=ax, color='steelblue', alpha=0.7, edgecolor='black')
    
    # Calculate statistics
    skew_val = skew(data)
    mean_val = np.mean(data)
    median_val = np.median(data)
    
    # Add lines
    ax.axvline(mean_val, color='red', linestyle='--', linewidth=2, alpha=0.7)
    ax.axvline(median_val, color='green', linestyle='--', linewidth=2, alpha=0.7)
    
    # Color title based on skewness improvement
    if abs(skew_val) < 0.5:
        title_color = 'green'
        status = '✅ Good'
    elif abs(skew_val) < 1:
        title_color = 'orange'
        status = '⚠️ Moderate'
    else:
        title_color = 'red'
        status = '❌ High'
    
    ax.set_title(f'{title}\nSkew: {skew_val:.3f} {status}', 
                 fontsize=11, fontweight='bold', color=title_color)
    ax.set_xlabel('')
    ax.grid(axis='y', alpha=0.3)

# Remove empty subplot
if len(transformations) < len(axes):
    fig.delaxes(axes[-1])

plt.suptitle(f'🔄 Transformation Comparison: {most_skewed_feature.replace("_", " ").title()}\n(Red=Mean, Green=Median)', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('💡 Insight: Box-Cox and Yeo-Johnson often provide optimal transformations by finding best lambda!')


In [ ]:
# Apply optimal transformations to all highly skewed features
print('=' * 70)
print('🔄 APPLYING OPTIMAL TRANSFORMATIONS TO ALL FEATURES')
print('=' * 70)

df_transformed = df.copy()
transformation_log = []

for feature in numerical_features:
    original_skew = skew(df[feature])
    
    # Skip if already approximately normal
    if abs(original_skew) < 0.5:
        transformation_log.append({
            'Feature': feature,
            'Original_Skew': round(original_skew, 3),
            'Applied_Transform': 'None (already symmetric)',
            'New_Skew': round(original_skew, 3),
            'Improvement': 0
        })
        continue
    
    # Try different transformations and pick best
    transforms = apply_transformations(df[feature])
    
    # Find transformation with skewness closest to 0
    best_transform = 'original'
    best_skew = abs(original_skew)
    
    for t_name in ['log', 'sqrt', 'boxcox', 'yeojohnson']:
        if t_name in transforms:
            t_skew = abs(skew(transforms[t_name]))
            if t_skew < best_skew:
                best_skew = t_skew
                best_transform = t_name
    
    # Apply best transformation
    if best_transform != 'original':
        df_transformed[f'{feature}_transformed'] = transforms[best_transform]
        df_transformed.drop(feature, axis=1, inplace=True)
        new_skew = skew(transforms[best_transform])
    else:
        new_skew = original_skew
    
    transformation_log.append({
        'Feature': feature,
        'Original_Skew': round(original_skew, 3),
        'Applied_Transform': best_transform,
        'New_Skew': round(new_skew, 3),
        'Improvement': round(abs(original_skew) - abs(new_skew), 3)
    })

transformation_df = pd.DataFrame(transformation_log)
transformation_df = transformation_df.sort_values('Improvement', ascending=False)

print('\n📊 Transformation Summary:')
print(transformation_df.to_string(index=False))

improved_features = transformation_df[transformation_df['Improvement'] > 0.3]
print(f'\n✅ Significant improvements (reduction > 0.3): {len(improved_features)} features')
if len(improved_features) > 0:
    print(improved_features[['Feature', 'Applied_Transform', 'Improvement']].to_string(index=False))


## 8. Analyzing Distribution of Target Variable 🎯

Target variable distribution is crucial for regression performance.


In [ ]:
# Target variable distribution analysis
print('=' * 70)
print('🎯 TARGET VARIABLE DISTRIBUTION ANALYSIS')
print('=' * 70)

target_skew = skew(df['target'])
target_kurt = kurtosis(df['target'])

print(f'Target variable statistics:')
print(f'   Mean: {df["target"].mean():.2f}')
print(f'   Median: {df["target"].median():.2f}')
print(f'   Std: {df["target"].std():.2f}')
print(f'   Skewness: {target_skew:.3f} ({"symmetric" if abs(target_skew) < 0.5 else "skewed"})')
print(f'   Kurtosis: {target_kurt:.3f}')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Original distribution
ax1 = axes[0, 0]
sns.histplot(df['target'], kde=True, ax=ax1, color='steelblue', alpha=0.7, edgecolor='black')
ax1.axvline(df['target'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["target"].mean():.0f}')
ax1.axvline(df['target'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {df["target"].median():.0f}')
ax1.set_title(f'Original Target Distribution\nSkewness: {target_skew:.3f}', fontsize=11, fontweight='bold')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Log-transformed target
ax2 = axes[0, 1]
target_log = np.log(df['target'] - df['target'].min() + 1)  # Handle negative if any
sns.histplot(target_log, kde=True, ax=ax2, color='forestgreen', alpha=0.7, edgecolor='black')
ax2.set_title(f'Log-Transformed Target\nSkewness: {skew(target_log):.3f}', fontsize=11, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# Box-Cox transformed target
ax3 = axes[1, 0]
# Shift to positive for Box-Cox
target_shifted = df['target'] - df['target'].min() + 1
target_boxcox, target_lambda = boxcox(target_shifted)
sns.histplot(target_boxcox, kde=True, ax=ax3, color='coral', alpha=0.7, edgecolor='black')
ax3.set_title(f'Box-Cox Transformed Target (λ={target_lambda:.3f})\nSkewness: {skew(target_boxcox):.3f}', 
              fontsize=11, fontweight='bold')
ax3.grid(axis='y', alpha=0.3)

# Q-Q plot for original target
ax4 = axes[1, 1]
stats.probplot(df['target'], dist='norm', plot=ax4)
ax4.set_title('Q-Q Plot: Original Target\n(Deviations = Non-normality)', fontsize=11, fontweight='bold')
ax4.grid(alpha=0.3)

plt.suptitle('🎯 Target Variable Distribution Analysis\n(Regression models perform better with symmetric targets!)', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'\n💡 Recommendation for target variable:')
if abs(target_skew) > 1:
    print(f'   ⚠️ Target is highly skewed. Consider log or Box-Cox transformation before modeling.')
elif abs(target_skew) > 0.5:
    print(f'   ⚠️ Target is moderately skewed. Transformation may improve results.')
else:
    print(f'   ✅ Target is approximately symmetric. No transformation needed.')


## 9. Key Insights and Preprocessing Recommendations 🎯

### 📋 Summary of Findings

#### Distribution Analysis Results:
1. **Highly Skewed Features**: Income, transaction amount, and website visits showed extreme positive skew (>2)
2. **Moderately Skewed**: House price, distance to city, and job tenure require attention
3. **Approximately Normal**: Age and credit score are well-behaved
4. **Target Variable**: May require transformation depending on skewness level

### 🎯 Preprocessing Recommendations

| Feature | Skewness | Recommended Transform | Rationale |
|---------|----------|---------------------|-----------|
| Income | High positive | Log or Box-Cox | Financial data typical pattern |
| Transaction amount | High positive | Log | Extreme outliers present |
| House price | Moderate positive | Log or Box-Cox | Real estate standard practice |
| Distance to city | Moderate positive | Square root | Milder transformation sufficient |
| Age | Low | None | Already approximately normal |
| Credit score | Low negative | None | Acceptable for modeling |

### 🔄 Transformation Selection Guide

```
High Positive Skew (> 1.5):
   ├── Try Box-Cox first (optimal lambda)
   └── Fallback to Log if Box-Cox fails

Moderate Positive Skew (0.5 - 1.5):
   ├── Square root (gentler than log)
   └── Log if sqrt insufficient

Negative Skew:
   ├── Reflect then apply positive skew transforms
   └── Or use Yeo-Johnson (handles negative directly)

Mixed/Unknown:
   └── Yeo-Johnson (most flexible, handles all values)
```

### ✅ Final Checklist Before Modeling

- [ ] Calculate skewness for all numerical features
- [ ] Visualize distributions with histograms/Q-Q plots
- [ ] Apply appropriate transformations to |skew| > 0.5 features
- [ ] Verify transformations improved normality
- [ ] Analyze target variable distribution
- [ ] Document all transformations for reproducibility
- [ ] Validate on holdout set with transformed features

**💡 Remember**: Always compare model performance with and without transformations—statistics guide us, but validation confirms!


## 🛠️ Hands-On Exercises

Apply your distribution analysis skills with these progressive exercises!

### Exercise 1: Basic Skewness Calculation
Calculate the skewness for all features in the dataset and create a DataFrame showing feature name, skewness value, and interpretation (symmetric, moderate, high).


In [ ]:
# Exercise 1: Your code here



### Exercise 2: Distribution Visualization
Create a histogram with KDE for the `income` feature. Add vertical lines for mean and median. Calculate and display the skewness value in the title.


In [ ]:
# Exercise 2: Your code here



### Exercise 3: Q-Q Plot Interpretation
Create a Q-Q plot for the `transaction_amount` feature. Based on the curve shape, determine if the distribution is positively skewed, negatively skewed, or has heavy tails.


In [ ]:
# Exercise 3: Your code here



### Exercise 4: Transformation Comparison
Apply log, square root, and Box-Cox transformations to the `house_price` feature. Calculate skewness for each and determine which transformation works best.


In [ ]:
# Exercise 4: Your code here



### Exercise 5: Before vs After Visualization
Create side-by-side histograms showing the original and transformed (best from Exercise 4) distributions of `house_price`. Include skewness values in titles.


In [ ]:
# Exercise 5: Your code here



### Exercise 6: Target Variable Analysis
Analyze the `target` variable distribution. If it is skewed, apply an appropriate transformation and compare the before/after Q-Q plots.


In [ ]:
# Exercise 6: Your code here



### Exercise 7: Correlation Impact Analysis
Calculate the correlation between `income` and `target` using (a) original values and (b) log-transformed income. How does the correlation change? What does this suggest?


In [ ]:
# Exercise 7: Your code here



### Exercise 8: Feature Selection by Skewness
Identify all features with |skewness| > 1.0. For each, determine the optimal transformation (the one that reduces absolute skewness the most). Create a summary table.


In [ ]:
# Exercise 8: Your code here



### Exercise 9: Reusable Skewness Function
Write a function `analyze_skewness(df, threshold=0.5)` that returns a DataFrame with all features, their skewness, recommended transformation, and expected improvement. Test it on this dataset.


In [ ]:
# Exercise 9: Your code here



### Exercise 10: Preprocessing Pipeline Recommendation
Based on all analysis, write a 3-paragraph recommendation for preprocessing this dataset for a linear regression model. Include which features to transform, which transformations to use, and how to validate the improvements.


In [ ]:
# Exercise 10: Write your recommendation as comments or print statements



## Solutions & Key Insights (Review After Attempting) 🔑

<details>
<summary>Click to expand solutions after attempting the exercises</summary>

### Exercise 1 Solution
```python
skew_data = []
for col in df.select_dtypes(include=[np.number]).columns:
    if col != 'target':
        s = skew(df[col])
        if abs(s) < 0.5:
            interp = 'symmetric'
        elif abs(s) < 1:
            interp = 'moderate'
        else:
            interp = 'high'
        skew_data.append({'Feature': col, 'Skewness': round(s, 3), 'Interpretation': interp})
skew_df = pd.DataFrame(skew_data)
print(skew_df.sort_values('Skewness', key=abs, ascending=False).to_string(index=False))
```
**Insight**: Sorting by absolute skewness helps prioritize which features need transformation first.

### Exercise 2 Solution
```python
plt.figure(figsize=(10, 6))
sns.histplot(df['income'], kde=True, color='steelblue')
plt.axvline(df['income'].mean(), color='red', linestyle='--', label=f'Mean: {df["income"].mean():.0f}')
plt.axvline(df['income'].median(), color='green', linestyle='--', label=f'Median: {df["income"].median():.0f}')
plt.title(f'Income Distribution (Skewness: {skew(df["income"]):.3f})')
plt.legend()
plt.show()
```
**Insight**: Large gap between mean and median confirms positive skew—transformation recommended.

### Exercise 3 Solution
```python
fig, ax = plt.subplots(figsize=(8, 6))
stats.probplot(df['transaction_amount'], dist='norm', plot=ax)
plt.title('Q-Q Plot: Transaction Amount')
plt.show()
# Upward curve at right end indicates positive skew (heavy right tail)
```
**Insight**: Upward curve at right indicates positive skew; downward curve would indicate negative skew.

### Exercise 4 Solution
```python
original_skew = skew(df['house_price'])
log_skew = skew(np.log(df['house_price']))
sqrt_skew = skew(np.sqrt(df['house_price']))
boxcox_data, _ = boxcox(df['house_price'])
boxcox_skew = skew(boxcox_data)

print(f'Original: {original_skew:.3f}')
print(f'Log: {log_skew:.3f}')
print(f'Sqrt: {sqrt_skew:.3f}')
print(f'Box-Cox: {boxcox_skew:.3f}')
# Box-Cox typically wins as it finds optimal lambda
```
**Insight**: Box-Cox usually provides optimal results by automatically finding the best transformation parameter.

### Exercise 5 Solution
```python
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['house_price'], kde=True, ax=ax1)
ax1.set_title(f'Original (Skew: {skew(df["house_price"]):.3f})')
sns.histplot(boxcox_data, kde=True, ax=ax2)
ax2.set_title(f'Box-Cox Transformed (Skew: {skew(boxcox_data):.3f})')
plt.tight_layout()
plt.show()
```
**Insight**: Visual comparison confirms transformation effectiveness—look for more symmetric shape.

### Exercise 6 Solution
```python
target_s = skew(df['target'])
print(f'Target skewness: {target_s:.3f}')
if abs(target_s) > 0.5:
    target_transformed, lam = boxcox(df['target'] - df['target'].min() + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    stats.probplot(df['target'], dist='norm', plot=ax1)
    ax1.set_title('Original Target Q-Q')
    stats.probplot(target_transformed, dist='norm', plot=ax2)
    ax2.set_title('Transformed Target Q-Q')
    plt.show()
```
**Insight**: Transformed target Q-Q plot should show points closer to diagonal line.

### Exercise 7 Solution
```python
corr_original = df['income'].corr(df['target'])
corr_log = np.log(df['income']).corr(df['target'])
print(f'Original correlation: {corr_original:.3f}')
print(f'Log-transformed correlation: {corr_log:.3f}')
# Log transformation often strengthens relationship by reducing outlier influence
```
**Insight**: Transformation can increase correlation by reducing outlier dominance in covariance calculation.

### Exercise 8 Solution
```python
high_skew_features = []
for col in numerical_features:
    if abs(skew(df[col])) > 1.0:
        transforms = {
            'log': abs(skew(np.log(df[col] + 1))),
            'sqrt': abs(skew(np.sqrt(df[col] + 1))),
            'boxcox': abs(skew(boxcox(df[col] + 1)[0])) if df[col].min() > -1 else float('inf')
        }
        best = min(transforms, key=transforms.get)
        high_skew_features.append({
            'Feature': col,
            'Original_Skew': skew(df[col]),
            'Best_Transform': best,
            'New_Skew': transforms[best]
        })
summary = pd.DataFrame(high_skew_features)
print(summary.to_string(index=False))
```
**Insight**: Systematic comparison ensures optimal transformation selection for each feature.

### Exercise 9 Solution
Your function should iterate through numerical columns, calculate skewness, determine if transformation needed based on threshold, test available transformations, and return DataFrame with recommendations. Include logic to handle negative values (use Yeo-Johnson) and zeros (add offset).

### Exercise 10 Solution
**Paragraph 1**: For linear regression, transform all features with |skew| > 0.5. Apply Box-Cox to house_price, income, and transaction_amount. Use square root for distance_to_city and job_tenure. Leave age and credit_score untransformed as they are approximately normal.

**Paragraph 2**: Apply Box-Cox to target variable if |skew| > 0.5, or log transformation if minimum value allows. This ensures homoscedasticity (constant variance of residuals) which is critical for valid hypothesis testing and prediction intervals in linear regression.

**Paragraph 3**: Validate by comparing model performance (RMSE, R²) on 5-fold cross-validation with and without transformations. Monitor residual plots for normality. Document all lambda parameters from Box-Cox for reproducibility on new data. Consider creating a preprocessing pipeline using sklearn's PowerTransformer for production deployment.

</details>
